# Exploratory Data Analysis

Produces class-balance and annotation-size summaries from the prepared dataset.

In [ ]:
from pathlib import Path
import sys
from collections import Counter
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from weapon_threat_detection.artifacts import utc_timestamp, write_json
from weapon_threat_detection.config import load_config

config = load_config(ROOT / 'configs' / 'project.yaml')
classes = config['dataset']['class_names']
labels_root = ROOT / config['project']['processed_data_dir'] / 'yolo_v1'
counts = Counter()
for label_file in labels_root.glob('*/labels/*.txt'):
    for line in label_file.read_text(encoding='utf-8').splitlines():
        if line.strip(): counts[classes[int(line.split()[0])]] += 1
report_dir = ROOT / config['project']['reports_dir']; report_dir.mkdir(exist_ok=True)
figure, axis = plt.subplots(figsize=(11, 5))
axis.bar(classes, [counts[name] for name in classes]); axis.tick_params(axis='x', rotation=35); axis.set_ylabel('Annotations'); figure.tight_layout()
plot_path = report_dir / f'class_distribution_{utc_timestamp()}.png'; figure.savefig(plot_path, dpi=180)
summary_path = write_json(report_dir / f'eda_summary_{utc_timestamp()}.json', {'class_counts': dict(counts), 'plot': str(plot_path)})
print(summary_path)
display(figure)